# 보이스피싱 사건 분리·화자분리·전사 (Colab GPU)

이 노트북은 OpenAI/GPT API를 사용하지 않습니다. Whisper와 pyannote를 Colab GPU에서 실행하며 결과는 Google Drive에 저장합니다. 런타임이 끊겨도 기존 `cases.json`이 있는 파일은 건너뛰므로 같은 전체 실행 셀을 다시 실행하면 이어서 처리됩니다.

## 시작 전 준비

1. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다.
2. Google Drive의 `내 드라이브/보이스피싱_분석/원본 영상 및 음원`에 세 분류 폴더를 통째로 업로드합니다.
3. Colab 왼쪽 열쇠(Secrets)에 `HF_TOKEN`이라는 이름으로 Hugging Face 읽기 토큰을 등록하고 이 노트북의 접근을 허용합니다.
4. pyannote 모델 이용조건은 이미 승인한 동일 계정의 토큰을 사용합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/보이스피싱_분석')
INPUT_ROOT = PROJECT_ROOT / '원본 영상 및 음원'
OUTPUT_ROOT = PROJECT_ROOT / '분석 결과'
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('입력:', INPUT_ROOT)
print('출력:', OUTPUT_ROOT)

In [ ]:
!nvidia-smi
!pip -q install faster-whisper==1.2.1 pyannote.audio==4.0.7

import torch
assert torch.cuda.is_available(), 'GPU가 선택되지 않았습니다. 런타임 유형을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'Colab Secrets에 HF_TOKEN을 등록하세요.'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
print('Hugging Face 토큰 확인 완료 (값은 표시하지 않음)')

## 분석 스크립트 업로드

다음 셀을 실행하고 이 노트북과 함께 제공된 `transcribe_cases.py`를 선택합니다. 스크립트는 Drive 프로젝트 폴더에 저장되어 다음 세션에도 유지됩니다. 이미 저장돼 있으면 업로드를 건너뜁니다.

In [ ]:
from google.colab import files
SCRIPT = PROJECT_ROOT / 'transcribe_cases.py'
if not SCRIPT.exists():
    uploaded = files.upload()
    assert 'transcribe_cases.py' in uploaded, 'transcribe_cases.py를 선택해야 합니다.'
    SCRIPT.write_bytes(uploaded['transcribe_cases.py'])
print('스크립트:', SCRIPT)
assert INPUT_ROOT.exists(), f'입력 폴더가 없습니다: {INPUT_ROOT}'
media = [p for p in INPUT_ROOT.rglob('*') if p.suffix.lower() in {'.mp3', '.mp4'}]
print('입력 미디어:', len(media), '개')
assert media, 'Google Drive에 원본 MP3/MP4를 먼저 업로드하세요.'

## 1건 시험 실행

먼저 검증된 34초 표본으로 GPU·전사·화자분리·역할분류가 모두 동작하는지 확인합니다.

In [ ]:
import subprocess, sys
sample_cmd = [
    sys.executable, str(SCRIPT),
    '--input', str(INPUT_ROOT), '--output', str(OUTPUT_ROOT),
    '--match', '36366_', '--limit', '1', '--force',
    '--device', 'cuda', '--compute-type', 'float16',
]
subprocess.run(sample_cmd, check=True)

In [ ]:
import csv, json
sample_json = next(OUTPUT_ROOT.rglob('36366_*/cases.json'))
data = json.loads(sample_json.read_text(encoding='utf-8'))
turns = data['cases'][0]['turns']
speakers = sorted({t['speaker_id'] for t in turns})
roles = sorted({t['role'] for t in turns if t['role']})
assert data['case_count'] >= 1 and len(speakers) >= 2
assert 'OFFENDER' in roles and 'VICTIM' in roles
assert all(t['text'].strip() for t in turns)
print('시험 정상:', {'사건': data['case_count'], '발화': len(turns), '화자': speakers, '역할': roles})
print((sample_json.parent / 'review.txt').read_text(encoding='utf-8'))

## 전체 513개 실행

아래 셀은 오래 실행됩니다. Colab 창을 열어 두세요. 세션이 중단되면 같은 셀을 다시 실행하면 완료된 파일을 건너뛰고 이어서 처리합니다.

In [ ]:
full_cmd = [
    sys.executable, str(SCRIPT),
    '--input', str(INPUT_ROOT), '--output', str(OUTPUT_ROOT),
    '--device', 'cuda', '--compute-type', 'float16',
]
subprocess.run(full_cmd, check=True)

## 완료 검증

입력과 결과 수, JSON 파싱, 빈 전사, 화자 미분리, 역할별 학습용/검수용 CSV를 검사합니다.

In [ ]:
from collections import Counter
media = [p for p in INPUT_ROOT.rglob('*') if p.suffix.lower() in {'.mp3', '.mp4'}]
json_files = list(OUTPUT_ROOT.rglob('cases.json'))
errors, empty, unresolved = [], [], []
case_total = turn_total = training_total = review_total = 0
role_counts = Counter()
for path in json_files:
    try:
        item = json.loads(path.read_text(encoding='utf-8'))
        case_total += item['case_count']
        for case in item['cases']:
            turn_total += len(case['turns'])
            if not case['turns'] or any(not t['text'].strip() for t in case['turns']): empty.append(str(path))
            for turn in case['turns']:
                role_counts[turn.get('role') or 'REVIEW'] += 1
                if not turn.get('speaker_id'): unresolved.append(str(path))
        with (path.parent / 'turns.csv').open(encoding='utf-8-sig') as f: training_total += sum(1 for _ in csv.DictReader(f))
        with (path.parent / '검수필요.csv').open(encoding='utf-8-sig') as f: review_total += sum(1 for _ in csv.DictReader(f))
    except Exception as exc: errors.append((str(path), repr(exc)))
summary = {
    '입력파일': len(media), '완료파일': len(json_files), '누락': len(media)-len(json_files),
    '사건후보': case_total, '전체발화': turn_total, '학습용발화': training_total,
    '검수필요발화': review_total, '역할': dict(role_counts),
    'JSON오류': len(errors), '빈전사파일': len(set(empty)), '화자미분리파일': len(set(unresolved)),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert not errors, errors[:5]
assert len(json_files) == len(media), '일부 파일이 미완료입니다. 전체 실행 셀을 다시 실행하세요.'